# VIIRS Flood Data — CLI Showcase

This notebook demonstrates how the **Atlantis** CLI fetches and processes [VIIRS](https://www.earthdata.nasa.gov/sensors/viirs) (Visible Infrared Imaging Radiometer Suite) satellite data for flood events catalogued in the [KuroSiwo](https://github.com/Cervest/kurosiwo) dataset.

## What does VIIRS provide?

| Property | Value |
|----------|-------|
| Sensor | VIIRS (Suomi-NPP / NOAA-20) |
| Product | 1-day composite flood map |
| Spatial resolution | ~375 m |
| CRS | EPSG:4326 (geographic lat/lon) |
| Pixel encoding | Integer codes (0–255) encoding land, water, cloud, flood probability |
| Source | NOAA JPSS public S3 bucket (`noaa-jpss`) or GMU legacy server |

## Notebook workflow

1. **Setup** — configure paths, parameters, imports
2. **Metadata** — build a metadata CSV from the KuroSiwo catalogue via the CLI
3. **Event selection** — pick one flood case and explore its spatial extent
4. **VIIRS fetch** — run the CLI to download & mosaic VIIRS tiles for that event
5. **File inspection** — examine the output GeoTIFF properties (CRS, resolution, extent)
6. **Visualisation** — plot the raw VIIRS raster and overlay on the KuroSiwo tiles

### Prerequisites

```bash
uv sync --extra notebooks    # install dependencies
git lfs pull                 # pull large assets (ks_catalogue.gpkg)
```

## 0. Setup

Define repository paths, select a flood case, and configure the VIIRS backend.
All CLI commands are printed before execution so you can reproduce them outside the notebook.

In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('Run this notebook from inside the Atlantis repository.')


def relpath_for_cli(path: Path) -> str:
    try:
        return str(path.relative_to(repo_root))
    except ValueError:
        return str(path)


def display_cli_command(command: list[str]) -> None:
    display(Markdown('### CLI command\n\n```bash\n' + shlex.join(command) + '\n```'))


repo_root = find_repo_root(Path.cwd().resolve())
src_dir = repo_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

drafts_dir = repo_root / 'notebooks' / 'drafts'
scratch_dir = drafts_dir / 'data' / 'viirs_showcase'
scratch_dir.mkdir(parents=True, exist_ok=True)

catalogue_path = repo_root / 'assets' / 'ks_catalogue.gpkg'
selected_case = 'KuroSiwo_1111004'
days_before = 0
days_after = 0
viirs_backend = 'noaa_s3'
viirs_format = 'tif'
refresh_metadata_cache = False
metadata_cache_tag = f'{viirs_backend}_{viirs_format}_before_{days_before}_after_{days_after}'
metadata_output_path = scratch_dir / 'kurosiwo_metadata_cli.csv'
fetch_root = scratch_dir / selected_case / 'viirs'

print(f'repo_root: {repo_root}')
print(f'catalogue_path: {catalogue_path}')
print(f'scratch_dir: {scratch_dir}')
print(f'selected_case: {selected_case}')
print(f'viirs_backend: {viirs_backend}')
print(f'viirs_format: {viirs_format}')
print(f'metadata_output_path: {metadata_output_path}')
print(f'fetch_root: {fetch_root}')
print(f'refresh_metadata_cache: {refresh_metadata_cache}')

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray as rxr
from shapely.geometry import box

plt.style.use('seaborn-v0_8-whitegrid')
pd.options.display.max_columns = 20
pd.options.display.width = 140

### VIIRS AOI grid bootstrap

The VIIRS fetcher needs a global grid of Areas of Interest (AOIs) to know which
satellite tiles to download. Each AOI is a 15° × 15° tile identified by an
integer `AOI_ID` that maps to filenames on the NOAA S3 bucket.

The grid is derived from a Natural Earth land mask — water-only tiles are
skipped, and a handful of special tiles are added manually. The result is saved
to `src/atlantis/fetchers/viirs/data/viirs_aois.geojson`, which is the path the
`VIIRSFetcher` reads at runtime.

> This grid can also be generated outside the notebook:  
> `uv run python scripts/setup.py`  
> That script is the canonical bootstrap for the whole repo and will grow to
> cover other data sources (GFM, MODIS, …).

> This cell is idempotent — it skips generation if the file already exists.

In [ ]:
import cartopy.io.shapereader as shpreader

aoi_grid_path = Path(repo_root, "src", "atlantis", "fetchers", "viirs", "data", "viirs_aois.geojson")
print(f"Target: {aoi_grid_path.relative_to(repo_root)}")

if aoi_grid_path.exists():
    print(f'AOI grid already exists: {aoi_grid_path.relative_to(repo_root)}')
    gdf_aois = gpd.read_file(aoi_grid_path)
    print(f'  {len(gdf_aois)} AOIs loaded')
else:
    print('Generating VIIRS AOI grid from Natural Earth land mask …')

    # Load land mask
    shpfilename = shpreader.natural_earth(resolution='110m', category='physical', name='land')
    land = gpd.read_file(shpfilename).to_crs('EPSG:4326')
    land_union = land.geometry.union_all()

    aois = []
    aoi_id = 1
    skipped = 0

    # Block 1: Americas + Atlantic (15° grid, lat 75→15)
    lon_edges = [-180, -165, -150, -135, -120, -105, -90, -75, -60]
    lat_edges = [75, 60, 45, 30, 15]
    for lat_max, lat_min in zip(lat_edges[:-1], lat_edges[1:]):
        for lon_min, lon_max in zip(lon_edges[:-1], lon_edges[1:]):
            tile = box(lon_min, lat_min, lon_max, lat_max)
            if not tile.intersects(land_union):
                continue
            if aoi_id == 11 and skipped == 0:
                skipped = 1
                continue
            if aoi_id == 16:
                aoi_id = 17
            if aoi_id == 22 and skipped == 1:
                skipped = 2
                continue
            aois.append((aoi_id, tile))
            aoi_id += 1

    # Block 2: South America + Africa (lat 15→-60)
    lon_edges = [-105, -90, -75, -60, -45, -30]
    lat_edges = [15, 0, -15, -30, -45, -60]
    skipped = 0
    for lat_max, lat_min in zip(lat_edges[:-1], lat_edges[1:]):
        for lon_min, lon_max in zip(lon_edges[:-1], lon_edges[1:]):
            tile = box(lon_min, lat_min, lon_max, lat_max)
            if not tile.intersects(land_union):
                continue
            if aoi_id == 34 and skipped == 0:
                skipped = 1
                continue
            aois.append((aoi_id, tile))
            aoi_id += 1

    # Block 3: Europe + Northern Asia (lat 75→30)
    lon_edges = [-15, 0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 165, 180]
    lat_edges = [75, 60, 45, 30]
    skipped = 0
    for lat_max, lat_min in zip(lat_edges[:-1], lat_edges[1:]):
        for lon_min, lon_max in zip(lon_edges[:-1], lon_edges[1:]):
            tile = box(lon_min, lat_min, lon_max, lat_max)
            if not tile.intersects(land_union):
                continue
            if aoi_id == 50:
                aoi_id = 52
            aois.append((aoi_id, tile))
            aoi_id += 1

    # Block 4: Africa + Asia + Oceania (lat 30→-60)
    aoi_id = 82
    lon_edges = [-30, -15, 0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 165, 180]
    lat_edges = [30, 15, 0, -15, -30, -45, -60]
    skipped = 0
    for lat_max, lat_min in zip(lat_edges[:-1], lat_edges[1:]):
        for lon_min, lon_max in zip(lon_edges[:-1], lon_edges[1:]):
            tile = box(lon_min, lat_min, lon_max, lat_max)
            if not tile.intersects(land_union):
                continue
            if aoi_id == 108:
                aoi_id = 109
            if aoi_id == 113 and skipped == 0:
                skipped = 1
                continue
            if aoi_id == 113 and skipped == 1:
                skipped = 2
                continue
            if aoi_id == 121 and skipped == 2:
                skipped = 3
                continue
            if aoi_id == 128 and skipped == 3:
                skipped = 4
                continue
            aois.append((aoi_id, tile))
            aoi_id += 1

    # Manual additions for tiles skipped or missing in the grid sweep
    aois.append((16, box(-60, 45, -45, 60)))
    aois.append((50, box(90, 75, 105, 90)))
    aois.append((51, box(105, 75, 120, 90)))
    aois.append((81, box(150, 30, 165, 45)))
    aois.append((108, box(75, -15, 90, 0)))
    aois.append((129, box(-150, 45, -135, 60)))
    aois.append((130, box(-60, 60, -45, 75)))
    aois.append((131, box(-45, 60, -30, 75)))
    aois.append((132, box(-30, 60, -15, 75)))
    aois.append((133, box(-160, 15, -150, 25)))
    aois.append((134, box(150, -15, 165, 0)))
    aois.append((135, box(165, -30, 180, -15)))
    aois.append((136, box(-180, -20, -165, -5)))

    # Sort by AOI_ID and build GeoDataFrame
    aois = sorted(aois, key=lambda x: x[0])
    gdf_aois = gpd.GeoDataFrame(
        {'AOI_ID': [a[0] for a in aois]},
        geometry=[a[1] for a in aois],
        crs='EPSG:4326',
    )

    aoi_grid_path.parent.mkdir(parents=True, exist_ok=True)
    gdf_aois.to_file(aoi_grid_path, driver='GeoJSON')
    print(f'Saved {len(gdf_aois)} AOIs → {aoi_grid_path.relative_to(repo_root)}')

gdf_aois.head()

## 1. Build the KuroSiwo metadata CSV

The Atlantis CLI can derive a summary CSV from the KuroSiwo GeoPackage catalogue.
Each row corresponds to one flood event and includes spatial bounds, date range, and estimated flood extent.

> Set `refresh_metadata_cache = True` above if you want to regenerate the CSV instead of reusing the cached version.

In [ ]:
metadata_command = [
    'uv',
    'run',
    'atlantis',
    'build-kurosiwo-metadata',
    '--catalogue',
    str(catalogue_path),
    '--output',
    str(metadata_output_path),
]
metadata_command_preview = [
    'uv',
    'run',
    'atlantis',
    'build-kurosiwo-metadata',
    '--catalogue',
    relpath_for_cli(catalogue_path),
    '--output',
    relpath_for_cli(metadata_output_path),
]
display_cli_command(metadata_command_preview)

In [ ]:
catalogue = gpd.read_file(catalogue_path)
if refresh_metadata_cache or not metadata_output_path.exists():
    print('Running:', shlex.join(metadata_command_preview))
    subprocess.run(metadata_command, cwd=repo_root, check=True)
    cache_status = 'rebuilt via CLI'
else:
    cache_status = 'loaded from cache'

metadata = pd.read_csv(metadata_output_path, parse_dates=['date_start', 'date_end'])
print(f'catalogue rows: {len(catalogue):,}')
print(f'distinct events: {catalogue["actid"].nunique()}')
print(f'metadata cache: {metadata_output_path.relative_to(repo_root)} ({cache_status})')
metadata.head()

In [ ]:
top_events = metadata.nlargest(10, 'max_flood_extent_km2').sort_values('max_flood_extent_km2')

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_events['flood_case'], top_events['max_flood_extent_km2'], color='#2f6c8f')
ax.set_title('Largest KuroSiwo events by derived flood extent')
ax.set_xlabel('max_flood_extent_km2')
ax.set_ylabel('')
plt.show()

## 2. Select and inspect one KuroSiwo flood case

We pick a single event from the metadata and visualise its spatial footprint.
The KuroSiwo catalogue stores each event as a collection of 224×224-pixel SAR tiles (10 m GSD).
The orange bounding box below is the **event envelope** — the area we will query VIIRS for.

In [ ]:
selected_event = metadata.loc[metadata['flood_case'] == selected_case].iloc[0]
selected_actid = int(selected_case.split('_')[-1])
event_rows = catalogue.to_crs('EPSG:4326')
event_rows = event_rows[event_rows['actid'] == selected_actid].copy()
event_bbox = box(
    selected_event['lon_min'],
    selected_event['lat_min'],
    selected_event['lon_max'],
    selected_event['lat_max'],
)

selected_event[
    ['flood_case', 'date_start', 'date_end', 'max_flood_extent_km2']
]

In [ ]:
# What does the KuroSiwo catalogue contain for this event?
# - Blue rectangles: individual 224×224-pixel SAR tile footprints from the catalogue
#   (each tile is ~2.2 km × 2.2 km at 10 m GSD)
# - Orange box: the bounding box enclosing ALL tiles — this is the area we'll query VIIRS for

fig, ax = plt.subplots(figsize=(8, 8))
event_rows.boundary.plot(ax=ax, linewidth=0.3, color='#6baed6', alpha=0.7, label='SAR tile footprints')
gpd.GeoSeries([event_bbox], crs='EPSG:4326').boundary.plot(
    ax=ax, color='#d95f02', linewidth=2.5, label='Event bounding box (VIIRS query area)')
ax.legend(loc='upper left', fontsize=9)
ax.set_title(f'{selected_case}: {len(event_rows)} catalogue tiles → 1 VIIRS query box', fontsize=11)
ax.set_xlabel('Longitude (°)')
ax.set_ylabel('Latitude (°)')
plt.tight_layout()
plt.show()

print(f"Number of SAR tiles in catalogue for this event: {len(event_rows)}")
print(f"Bounding box (lon/lat): [{selected_event['lon_min']:.2f}, {selected_event['lat_min']:.2f}] → [{selected_event['lon_max']:.2f}, {selected_event['lat_max']:.2f}]")

### KuroSiwo catalogue: per-tile statistics

Before fetching VIIRS data, we look at what the KuroSiwo catalogue already records for this event.

Each tile has two labels derived from Sentinel-1 SAR:

| Label | Meaning |
|-------|---------|
| **pflood** | % of the 224×224 tile covered by flood water (ground-truth label) |
| **pwater** | % covered by permanent water bodies (lakes, rivers) — static across time |

We focus on **flood-time GRD tiles** (`master=True`, `crank=1`) so we can later compare what the high-res SAR-based catalogue says vs. what the coarser VIIRS composite shows.

In [ ]:
import numpy as np

# ── Tile-level statistics for the selected event ──────────────────────────────
flood_tiles = event_rows[(event_rows['master'] == True) & (event_rows['crank'] == 1)].copy()
pflood = flood_tiles['pflood'].dropna()
pwater = (
    flood_tiles['pwater'].dropna()
    if 'pwater' in flood_tiles.columns
    else pd.Series(dtype=float)
)

print(f'Event:                {selected_case}')
print(f'Flood-time GRD tiles: {len(flood_tiles)}')
print()
print('── pflood  (flood coverage per 224×224-px tile, 10 m GSD) ──')
print(f'  Non-NaN tiles:    {len(pflood)}  ({100 * len(pflood) / max(len(flood_tiles), 1):.0f} %)')
print(f'  Tiles with flood: {(pflood > 0).sum()}  ({100 * (pflood > 0).mean():.0f} %)')
print(f'  Range:            {pflood.min():.1f} % – {pflood.max():.1f} %')
print(f'  Mean ± std:       {pflood.mean():.1f} % ± {pflood.std():.1f} %')
if len(pwater):
    print()
    print('── pwater  (permanent water per tile) ──')
    print(f'  Non-NaN tiles:    {len(pwater)}  ({100 * len(pwater) / max(len(flood_tiles), 1):.0f} %)')
    print(f'  Tiles with water: {(pwater > 0).sum()}  ({100 * (pwater > 0).mean():.0f} %)')
    print(f'  Range:            {pwater.min():.1f} % – {pwater.max():.1f} %')
    print(f'  Mean ± std:       {pwater.mean():.1f} % ± {pwater.std():.1f} %')

# ── Distributions ──────────────────────────────────────────────────────────────
ncols = 3 if len(pwater) else 2
fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4))

# pflood histogram
ax = axes[0]
ax.hist(pflood, bins=20, color='#c0392b', edgecolor='black', alpha=0.75)
ax.axvline(pflood.median(), color='darkred', ls='--', lw=1.5,
           label=f'Median: {pflood.median():.1f} %')
ax.set_xlabel('pflood (%)')
ax.set_ylabel('Number of tiles')
ax.set_title('Flood coverage per tile (pflood)')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# pflood class-breakdown pie
ax = axes[1]
counts_pf = [
    (pflood == 0).sum(),
    ((pflood > 0) & (pflood <= 5)).sum(),
    ((pflood > 5) & (pflood <= 50)).sum(),
    (pflood > 50).sum(),
]
labels_pf = ['None (0 %)', 'Low (0–5 %)', 'Moderate (5–50 %)', 'High (>50 %)']
colors_pf = ['#2ecc71', '#f39c12', '#e74c3c', '#c0392b']
ax.pie(counts_pf, labels=labels_pf, autopct='%1.0f%%', colors=colors_pf, startangle=90)
ax.set_title('pflood class breakdown')

# pwater histogram (if available)
if len(pwater):
    ax = axes[2]
    nz = pwater[pwater > 0]
    if len(nz):
        ax.hist(nz, bins=20, color='#2980b9', edgecolor='black', alpha=0.75)
        ax.set_xlabel('pwater (%)')
        ax.set_ylabel('Number of tiles (pwater > 0)')
        ax.set_title('Permanent water per tile (pwater, non-zero)')
        ax.grid(axis='y', alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'pwater = 0 for all tiles', ha='center', va='center',
                transform=ax.transAxes, fontsize=10)
        ax.set_title('pwater (all tiles dry)')

plt.suptitle(f'{selected_case}: KuroSiwo flood-time tile statistics', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Tile footprint maps ────────────────────────────────────────────────────────
# Why do most tiles appear gray?
# → The vast majority of tiles have pflood=0 (no flood detected by SAR).
#   Only a small cluster of tiles actually contained flood water.
#   The coloured tiles are the ones where the ML model detected flooding.
#   We zoom into the flooded region below for clarity.

has_pwater_map = len(pwater) > 0 and flood_tiles['pwater'].notna().any()
ncols_map = 2 if has_pwater_map else 1
fig, axes_map = plt.subplots(1, ncols_map, figsize=(7 * ncols_map, 6))
if ncols_map == 1:
    axes_map = [axes_map]

# Plot ALL tiles (pflood)
flood_tiles.plot(
    ax=axes_map[0], column='pflood', cmap='YlOrRd',
    linewidth=0.25, edgecolor='gray', vmin=0, vmax=100, legend=True,
    missing_kwds={'color': 'lightgray', 'label': 'No flood label (NaN)'},
    legend_kwds={'label': 'pflood (%)', 'shrink': 0.7},
)
axes_map[0].set_title(f'All {len(flood_tiles)} flood-time tiles: pflood (%)\n(gray = 0% or NaN → no flood detected)', fontsize=9)
axes_map[0].set_xlabel('Longitude (°)')
axes_map[0].set_ylabel('Latitude (°)')

if has_pwater_map:
    flood_tiles.plot(
        ax=axes_map[1], column='pwater', cmap='Blues',
        linewidth=0.25, edgecolor='gray', vmin=0, vmax=100, legend=True,
        missing_kwds={'color': 'lightgray', 'label': 'No label (NaN)'},
        legend_kwds={'label': 'pwater (%)', 'shrink': 0.7},
    )
    axes_map[1].set_title(f'All {len(flood_tiles)} flood-time tiles: pwater (%)\n(gray = 0% or NaN → no permanent water)', fontsize=9)
    axes_map[1].set_xlabel('Longitude (°)')
    axes_map[1].set_ylabel('Latitude (°)')

plt.suptitle(f'{selected_case}: spatial distribution of catalogue labels', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Zoomed view: only tiles with pflood > 0 ────────────────────────────────────
flooded_tiles = flood_tiles[flood_tiles['pflood'] > 0]
if len(flooded_tiles) > 0:
    fig, ax_zoom = plt.subplots(figsize=(8, 6))
    # Show all tiles as background in light gray
    flood_tiles.plot(ax=ax_zoom, facecolor='#f0f0f0', edgecolor='#cccccc', linewidth=0.2)
    # Overlay only the flooded tiles with colour
    flooded_tiles.plot(
        ax=ax_zoom, column='pflood', cmap='YlOrRd',
        linewidth=0.5, edgecolor='black', vmin=0, vmax=100, legend=True,
        legend_kwds={'label': 'pflood (%)', 'shrink': 0.7},
    )
    # Zoom to the flooded region with some padding
    bounds = flooded_tiles.total_bounds  # [minx, miny, maxx, maxy]
    pad = 0.1 * max(bounds[2] - bounds[0], bounds[3] - bounds[1])
    ax_zoom.set_xlim(bounds[0] - pad, bounds[2] + pad)
    ax_zoom.set_ylim(bounds[1] - pad, bounds[3] + pad)
    ax_zoom.set_title(
        f'Zoomed: {len(flooded_tiles)} tiles with pflood > 0\n'
        f'(out of {len(flood_tiles)} total flood-time tiles)',
        fontsize=10, fontweight='bold')
    ax_zoom.set_xlabel('Longitude (°)')
    ax_zoom.set_ylabel('Latitude (°)')
    plt.tight_layout()
    plt.show()
else:
    print("No tiles with pflood > 0 found.")

## 3. Fetch VIIRS data via the CLI

The `fetch-kurosiwo-viirs` command:

1. Looks up the event's bounding box from the metadata
2. Queries the NOAA JPSS S3 bucket for all VIIRS 1-day composite tiles intersecting that box within the event's date window (`±days_before/after`)
3. Downloads matching tiles, mosaics them, and clips the result to the event extent
4. Writes the output GeoTIFF(s) to the processed directory

By default, Atlantis writes only the **raw** (unclassified) raster. Pass `--classify` to also produce flood-extent, quality-mask, and permanent-water layers.

In [ ]:
fetch_command = [
    'uv',
    'run',
    'atlantis',
    'fetch-kurosiwo-viirs',
    '--catalogue',
    str(catalogue_path),
    '--case',
    selected_case,
    '--output',
    str(scratch_dir),
    '--days-before',
    str(days_before),
    '--days-after',
    str(days_after),
    '--viirs-backend',
    viirs_backend,
    '--viirs-format',
    viirs_format,
]
fetch_command_preview = [
    'uv',
    'run',
    'atlantis',
    'fetch-kurosiwo-viirs',
    '--catalogue',
    relpath_for_cli(catalogue_path),
    '--case',
    selected_case,
    '--output',
    relpath_for_cli(scratch_dir),
    '--days-before',
    str(days_before),
    '--days-after',
    str(days_after),
    '--viirs-backend',
    viirs_backend,
    '--viirs-format',
    viirs_format,
]
display_cli_command(fetch_command_preview)


In [ ]:
print('Running:', shlex.join(fetch_command_preview))
subprocess.run(fetch_command, cwd=repo_root, check=True)
fetch_dir = fetch_root / 'processed'
if not fetch_dir.exists():
    raise RuntimeError(f'Expected processed output at {fetch_dir}')
sorted(path.name for path in fetch_dir.glob('*.tif'))


## 4. Inspect the fetched files

Before plotting, let's look at what the fetch step produced: file names, sizes, and raster metadata reported by GDAL.

In [ ]:
import os

fetch_dir = fetch_root / 'processed'
print(f"Output directory: {fetch_dir.relative_to(repo_root)}")
print(f"{'─' * 60}")

for tif in sorted(fetch_dir.glob('*.tif')):
    size_kb = tif.stat().st_size / 1024
    print(f"  {tif.name}  ({size_kb:.1f} KB)")

print(f"\n{'═' * 60}")
print("GDAL metadata for the raw VIIRS raster:")
print(f"{'═' * 60}\n")

# Use rasterio to display gdalinfo-like metadata
import rasterio
raw_path = sorted(fetch_dir.glob('*_raw.tif'))[0]
with rasterio.open(raw_path) as src:
    print(f"  Driver:      {src.driver}")
    print(f"  Dimensions:  {src.width} x {src.height} pixels, {src.count} band(s)")
    print(f"  Dtype:       {src.dtypes[0]}")
    print(f"  CRS:         {src.crs}")
    print(f"  Bounds:      {src.bounds}")
    print(f"  Resolution:  {abs(src.res[0]):.6f}° x {abs(src.res[1]):.6f}°")
    print(f"               ≈ {abs(src.res[0]) * 111_320:.0f} m x {abs(src.res[1]) * 111_320:.0f} m (at equator)")
    print(f"  NoData:      {src.nodata}")
    print(f"  Transform:   {src.transform}")

## 5. Visualise the raw VIIRS raster

The raw raster encodes per-pixel integer values (0–255) from the VIIRS 1-day flood composite.
We use the `turbo` colourmap to reveal the spatial structure across the event extent.

Key pixel codes in the VIIRS product:

| Code | Meaning |
|------|---------|
| 0 | No data / fill |
| 17 | Permanent water |
| 20 | Seasonal water |
| 30 | Cloud |
| 99 | Open water |
| ≥160 | Flood signal (higher = more confident) |

In [ ]:
# Load the raw GeoTIFF into an xarray DataArray
processed_files = sorted(fetch_dir.glob('*_raw.tif'))
if not processed_files:
    raise RuntimeError('No processed GeoTIFFs were written by the CLI fetch command.')

raw_da = rxr.open_rasterio(processed_files[0]).squeeze(drop=True)
dataset = {'raw': raw_da}

print(f"Loaded: {processed_files[0].name}")
print(f"Shape:  {raw_da.shape} (y × x pixels)")
print(f"Dtype:  {raw_da.dtype}")
print(f"Range:  [{int(raw_da.min())}, {int(raw_da.max())}]")
raw_da

In [ ]:
# Pixel-value histogram — understand what codes are present
vals = raw_da.values.ravel()
vals_nonzero = vals[vals > 0]  # exclude nodata / fill

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)

# Full histogram
ax1.hist(vals_nonzero, bins=50, color='#2f6c8f', edgecolor='black', alpha=0.8)
ax1.axvline(160, color='red', ls='--', lw=1.5, label='Flood threshold (≥160)')
ax1.set_xlabel('Pixel value')
ax1.set_ylabel('Count')
ax1.set_title('Distribution of non-zero pixel values')
ax1.legend()

# Unique values table
unique, counts = np.unique(vals_nonzero, return_counts=True)
ax2.bar(unique, counts, color='#2f6c8f', width=1.0)
ax2.axvline(160, color='red', ls='--', lw=1.5, label='Flood threshold')
ax2.set_xlabel('Pixel value')
ax2.set_ylabel('Count')
ax2.set_title('Counts per unique pixel code')
ax2.legend()

plt.suptitle(f'{selected_case}: VIIRS pixel value distribution', fontsize=11, fontweight='bold')
plt.show()

# Summary table of most common codes
print(f"\nTop 10 pixel values (excluding 0):")
order = np.argsort(-counts)
for i in order[:10]:
    pct = 100 * counts[i] / len(vals_nonzero)
    print(f"  Code {unique[i]:3d}: {counts[i]:6d} pixels ({pct:.1f}%)")

In [ ]:
# Spatial view of the raw VIIRS data with pixel code legend
from matplotlib.patches import Patch

# Key VIIRS pixel codes and their meanings
viirs_codes = {
    0: ('No data / fill', '#000000'),
    1: ('Land (no water)', '#8B4513'),
    17: ('Permanent water', '#1f77b4'),
    20: ('Seasonal water', '#17becf'),
    30: ('Cloud', '#cccccc'),
    99: ('Open water', '#4682B4'),
    160: ('Flood (low conf.)', '#FFFF00'),
    170: ('Flood (medium)', '#FFA500'),
    200: ('Flood (high conf.)', '#FF0000'),
}

fig, (ax, ax_leg) = plt.subplots(1, 2, figsize=(14, 7), gridspec_kw={'width_ratios': [3, 1]},
                                  constrained_layout=True)

# Map
dataset['raw'].plot(ax=ax, cmap='turbo', add_colorbar=True,
                    cbar_kwargs={'label': 'Pixel code', 'shrink': 0.8})
ax.set_title(f'{selected_case}: Raw VIIRS 1-day composite (375 m)', fontsize=11, fontweight='bold')
ax.set_xlabel('Longitude (°)')
ax.set_ylabel('Latitude (°)')

# Legend panel
legend_patches = [Patch(facecolor=color, edgecolor='black', linewidth=0.5,
                        label=f'{code}: {label}')
                  for code, (label, color) in viirs_codes.items()]
ax_leg.legend(handles=legend_patches, loc='center', fontsize=9,
              title='VIIRS pixel codes', title_fontsize=10,
              frameon=True, edgecolor='gray')
ax_leg.axis('off')

plt.show()

## 6. Overlay: VIIRS raster + KuroSiwo event tiles

This final plot compares the two sources side by side and overlaid:

- **Left**: KuroSiwo catalogue tile footprints (224×224 px SAR tiles at 10 m)
- **Centre**: VIIRS raw composite (375 m pixels) — shows what the satellite observed
- **Right**: Overlay of both — helps assess spatial correspondence between the SAR-derived labels and the VIIRS composite

In [ ]:
import numpy as np

# ── KuroSiwo event tiles (flood-time GRD) ────────────────────────────────────
flood_tiles_geo = event_rows[
    (event_rows['master'] == True) & (event_rows['crank'] == 1)
].copy()

# ── 3-panel figure ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: KuroSiwo event extent (tile footprints)
ax = axes[0]
flood_tiles_geo.plot(ax=ax, facecolor='#fed976', edgecolor='black', linewidth=0.3, alpha=0.7)
ax.set_title('KuroSiwo event extent\n(SAR tile footprints)', fontsize=10, fontweight='bold')
ax.set_xlabel('Longitude (°)')
ax.set_ylabel('Latitude (°)')

# Panel 2: VIIRS raw data (raster)
ax = axes[1]
dataset['raw'].plot(ax=ax, cmap='turbo', add_colorbar=True)
ax.set_title('VIIRS raw data\n(375 m per-pixel)', fontsize=10, fontweight='bold')
ax.set_xlabel('Longitude (°)')
ax.set_ylabel('')

# Panel 3: Overlay — VIIRS raster + KuroSiwo tile boundaries
ax = axes[2]
dataset['raw'].plot(ax=ax, cmap='turbo', add_colorbar=False)
flood_tiles_geo.boundary.plot(ax=ax, color='black', linewidth=0.3, alpha=0.5)
ax.set_title('Overlay\n(VIIRS raw data + KuroSiwo tiles)', fontsize=10, fontweight='bold')
ax.set_xlabel('Longitude (°)')
ax.set_ylabel('')

plt.suptitle(f'{selected_case} — EPSG:4326, no reprojection needed', fontsize=11)
plt.tight_layout()
plt.show()